# ESMT Rankings Intelligence

**GitHub Actions cron → live collection → CSV/state files → Streamlit**

To make failures easy to diagnose, only one technically simple source is enabled: the official Corporate Knights WordPress API. There is no paid AI, semantic search, ranking history, or complex scraping in this checkpoint.

Collection rules:

- first successful run: request the previous **90 days**;
- later runs: request records since the last successful run, with a **48-hour safety overlap**;
- deduplicate by canonical URL;
- retain stored records for **365 days**;
- make collection failure stop the workflow instead of silently reporting “no news”.


In [17]:
from pathlib import Path
from urllib.parse import urlencode, urlparse, urlunparse, parse_qsl
from urllib.request import Request, urlopen
from datetime import datetime, timezone
import hashlib
import html
import json
import re

import pandas as pd
import requests
from IPython.display import display


DATA_DIR = Path("data")
NEWS_PATH = DATA_DIR / "news.csv"
STATUS_PATH = DATA_DIR / "collector_status.csv"
STATE_PATH = DATA_DIR / "run_state.json"

INITIAL_BACKFILL_DAYS = 90
RETENTION_DAYS = 365
SAFETY_OVERLAP_HOURS = 48
REQUEST_TIMEOUT_SECONDS = 30
NOW_UTC = pd.Timestamp.now(tz="UTC")

SOURCE_REGISTRY = [
    {
        "source_id": "ft",
        "publisher": "Financial Times",
        "source_group": "ranking",
        "official_domains": ["ft.com", "rankings.ft.com"],
        "entrypoints": ["https://www.ft.com/business-education"],
        "feed_url": None,
        "collection_method": "html_hub",
        "allowed_path_regex": r"^/(business-education|content/|rankings/)",
    },
    {
        "source_id": "qs",
        "publisher": "QS",
        "source_group": "ranking",
        "official_domains": ["qs.com", "support.qs.com", "topmba.com", "topuniversities.com"],
        "entrypoints": ["https://www.qs.com/insights"],
        "feed_url": None,
        "collection_method": "qs_hub",
        "allowed_path_regex": r"^/(insights|hc/en-gb/articles|mba-rankings)",
    },
    {
        "source_id": "bloomberg",
        "publisher": "Bloomberg Businessweek",
        "source_group": "ranking",
        "official_domains": ["bloomberg.com"],
        "entrypoints": ["https://www.bloomberg.com/business-schools/"],
        "feed_url": None,
        "collection_method": "html_hub",
        "allowed_path_regex": r"^/business-schools/",
    },
    {
        "source_id": "corporate_knights",
        "publisher": "Corporate Knights",
        "source_group": "ranking",
        "official_domains": ["corporateknights.com"],
        "entrypoints": ["https://corporateknights.com/rankings/top-40-mba-rankings/"],
        "feed_url": "https://corporateknights.com/feed/",
        "api_url": "https://corporateknights.com/wp-json/wp/v2/posts",
        "collection_method": "wordpress_api",
        "allowed_path_regex": r"^/",
    },
    {
        "source_id": "poets_quants",
        "publisher": "Poets&Quants",
        "source_group": "ranking",
        "official_domains": ["poetsandquants.com"],
        "entrypoints": ["https://poetsandquants.com/"],
        "feed_url": "https://poetsandquants.com/feed/",
        "api_url": "https://poetsandquants.com/wp-json/wp/v2/posts",
        "collection_method": "wordpress_api",
        "allowed_path_regex": r"^/",
    },
    {
        "source_id": "ceoworld",
        "publisher": "CEOWORLD Magazine",
        "source_group": "ranking",
        "official_domains": ["ceoworld.biz"],
        "entrypoints": ["https://ceoworld.biz/"],
        "feed_url": "https://ceoworld.biz/feed/",
        "collection_method": "rss_paginated",
        "allowed_path_regex": r"^/",
    },
    {
        "source_id": "aacsb",
        "publisher": "AACSB",
        "source_group": "accreditation",
        "official_domains": ["aacsb.edu"],
        "entrypoints": ["https://www.aacsb.edu/media-center/news", "https://www.aacsb.edu/insights"],
        "feed_url": "https://www.aacsb.edu/sitemap.xml",
        "collection_method": "sitemap",
        "allowed_path_regex": r"^/(media-center/news|insights)/",
    },
    {
        "source_id": "amba",
        "publisher": "AMBA",
        "source_group": "accreditation",
        "official_domains": ["amba-bga.com"],
        "entrypoints": ["https://www.amba-bga.com/insights/search"],
        "feed_url": "https://www.amba-bga.com/sitemap.xml",
        "collection_method": "sitemap",
        "allowed_path_regex": r"^/insights/",
    },
    {
        "source_id": "equis",
        "publisher": "EQUIS / EFMD Global",
        "source_group": "accreditation",
        "official_domains": ["efmdglobal.org", "blog.efmdglobal.org"],
        "entrypoints": ["https://blog.efmdglobal.org/category/accreditations-assessments/equis/"],
        "feed_url": "https://blog.efmdglobal.org/feed/",
        "collection_method": "rss_paginated",
        "allowed_path_regex": r"^/",
    },
    {
        "source_id": "zeva",
        "publisher": "ZEvA",
        "source_group": "accreditation",
        "official_domains": ["zeva.org"],
        "entrypoints": ["https://www.zeva.org/"],
        "feed_url": "https://www.zeva.org/sitemap.xml",
        "collection_method": "sitemap",
        "allowed_path_regex": r"^/",
    },
    {
        "source_id": "wissenschaftsrat",
        "publisher": "Wissenschaftsrat",
        "source_group": "accreditation",
        "official_domains": ["wissenschaftsrat.de"],
        "entrypoints": ["https://www.wissenschaftsrat.de/DE/Aktuelles/Presse/Pressemitteilungen"],
        "feed_url": None,
        "collection_method": "html_hub",
        "allowed_path_regex": r"^/(DE/Aktuelles/Presse|SharedDocs/Pressemitteilungen)/",
    },
]

SOURCE = next(source for source in SOURCE_REGISTRY if source["source_id"] == "corporate_knights")
ADDITIONAL_SOURCES = [source for source in SOURCE_REGISTRY if source is not SOURCE]

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run time: {NOW_UTC.isoformat()}")
print(f"Registered sources: {len(SOURCE_REGISTRY)}")
for source in SOURCE_REGISTRY:
    print(f"Source: {source['publisher']} ({source['collection_method']})")


Run time: 2026-08-20T19:09:07.967243+00:00
Registered sources: 11
Source: Financial Times (html_hub)
Source: QS (qs_hub)
Source: Bloomberg Businessweek (html_hub)
Source: Corporate Knights (wordpress_api)
Source: Poets&Quants (wordpress_api)
Source: CEOWORLD Magazine (rss_paginated)
Source: AACSB (sitemap)
Source: AMBA (sitemap)
Source: EQUIS / EFMD Global (rss_paginated)
Source: ZEvA (sitemap)
Source: Wissenschaftsrat (html_hub)


In [18]:
sources_df = pd.DataFrame(SOURCE_REGISTRY)
display(sources_df[[
    "publisher", "source_group", "collection_method", "entrypoints"
]])


,publisher,source_group,collection_method,entrypoints
0,Financial Times,ranking,html_hub,[https://www.ft.com/business-education]
1,QS,ranking,qs_hub,[https://www.qs.com/insights]
2,Bloomberg Businessweek,ranking,html_hub,[https://www.bloomberg.com/business-schools/]
3,Corporate Knights,ranking,wordpress_api,[https://corporateknights.com/rankings/top-40-...
4,Poets&Quants,ranking,wordpress_api,[https://poetsandquants.com/]
5,CEOWORLD Magazine,ranking,rss_paginated,[https://ceoworld.biz/]
6,AACSB,accreditation,sitemap,"[https://www.aacsb.edu/media-center/news, http..."
7,AMBA,accreditation,sitemap,[https://www.amba-bga.com/insights/search]
8,EQUIS / EFMD Global,accreditation,rss_paginated,[https://blog.efmdglobal.org/category/accredit...
9,ZEvA,accreditation,sitemap,[https://www.zeva.org/]


## 1. Decide whether this is the first backfill or an incremental run

The state is stored in `data/run_state.json`. The scheduled workflow commits this file back to GitHub after every successful run, so the next run can continue from the recorded timestamp.


In [19]:
def load_state(path):
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return {}


state = load_state(STATE_PATH)
last_successful_value = state.get("last_successful_run_utc")
last_successful_run = (
    pd.to_datetime(last_successful_value, utc=True, errors="coerce")
    if last_successful_value
    else pd.NaT
)

first_run = pd.isna(last_successful_run)
if first_run:
    collection_since = NOW_UTC - pd.Timedelta(days=INITIAL_BACKFILL_DAYS)
    collection_mode = "initial_90_day_backfill"
else:
    collection_since = last_successful_run - pd.Timedelta(hours=SAFETY_OVERLAP_HOURS)
    collection_mode = "incremental_with_48h_overlap"

print(f"Mode: {collection_mode}")
print(f"Request publications after: {collection_since.isoformat()}")


Mode: incremental_with_48h_overlap
Request publications after: 2026-08-18T19:07:09.656939+00:00


## 2. Collect live records

The source is queried server-side with an `after` timestamp. Pagination is supported, but no browser automation or HTML-page crawling is needed.


In [20]:
from html.parser import HTMLParser
from urllib.parse import urlencode, urljoin, urlparse, urlunparse, parse_qsl
from xml.etree import ElementTree


TRACKING_KEYS = {
    "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
    "fbclid", "gclid", "mc_cid", "mc_eid",
}


class LinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []
        self._href = ""
        self._text = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() != "a":
            return
        attributes = dict(attrs)
        self._href = attributes.get("href", "")
        self._text = []

    def handle_data(self, data):
        if self._href:
            self._text.append(data)

    def handle_endtag(self, tag):
        if tag.lower() == "a" and self._href:
            self.links.append((self._href, " ".join(self._text)))
            self._href = ""
            self._text = []


def canonicalize_url(value):
    parsed = urlparse(str(value).strip())
    query = [
        (key, item)
        for key, item in parse_qsl(parsed.query, keep_blank_values=True)
        if key.lower() not in TRACKING_KEYS
    ]
    path = re.sub(r"/{2,}", "/", parsed.path or "/")
    return urlunparse((parsed.scheme.lower(), parsed.netloc.lower(), path, "", urlencode(query), ""))


def clean_html(value):
    text = re.sub(r"<[^>]+>", " ", value or "")
    return re.sub(r"\s+", " ", html.unescape(text)).strip()


def make_record(source, url, title, excerpt="", published_at=pd.NaT):
    return {
        "article_id": hashlib.sha1(url.encode("utf-8")).hexdigest()[:16],
        "source_id": source["source_id"],
        "publisher": source["publisher"],
        "source_group": source["source_group"],
        "title": title,
        "excerpt": excerpt,
        "url": url,
        "published_at": published_at,
        "collected_at": NOW_UTC,
    }


def source_url_is_allowed(source, url):
    parsed = urlparse(url)
    host = (parsed.hostname or "").lower()
    allowed_domains = tuple(source["official_domains"])
    allowed_path = re.compile(source.get("allowed_path_regex", r"^/"), re.I)
    return (
        parsed.scheme in {"http", "https"}
        and any(host == domain or host.endswith("." + domain) for domain in allowed_domains)
        and allowed_path.search(parsed.path) is not None
    )


def collect_wordpress_posts(source, since):
    base_params = {
        "per_page": 100,
        "after": since.isoformat(),
        "orderby": "date",
        "order": "desc",
        "_fields": "id,date_gmt,link,title,excerpt",
    }
    all_items = []
    page = 1
    total_pages = 1

    while page <= total_pages:
        request_url = f"{source['api_url']}?{urlencode(dict(base_params, page=page))}"
        request = Request(
            request_url,
            headers={"User-Agent": "ESMT-ranking-intelligence-cron-test/1.0", "Accept": "application/json"},
        )
        with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            all_items.extend(json.loads(response.read().decode("utf-8")))
            total_pages = int(response.headers.get("X-WP-TotalPages", "1"))
        page += 1

    records = []
    for item in all_items:
        url = canonicalize_url(item.get("link", ""))
        if not source_url_is_allowed(source, url):
            continue
        title = clean_html((item.get("title") or {}).get("rendered", ""))
        if not title or not url:
            continue
        records.append(make_record(
            source,
            url,
            title,
            clean_html((item.get("excerpt") or {}).get("rendered", "")),
            pd.to_datetime(item.get("date_gmt"), utc=True, errors="coerce"),
        ))

    return records


def collect_rss_source(source):
    response = requests.get(
        source["feed_url"],
        headers={"User-Agent": "Mozilla/5.0", "Accept": "application/rss+xml,application/xml"},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    root = ElementTree.fromstring(response.content)
    records = []

    for item in root.iter():
        children = {
            child.tag.rsplit("}", 1)[-1].lower(): child
            for child in list(item)
        }
        if "title" not in children or "link" not in children:
            continue
        title = clean_html(children["title"].text or "")
        url = canonicalize_url(children["link"].text or "")
        if not title or not url or not source_url_is_allowed(source, url):
            continue
        description = children.get("description") or children.get("summary")
        published = children.get("pubdate") or children.get("published") or children.get("updated")
        records.append(make_record(
            source,
            url,
            title,
            clean_html(description.text if description is not None else ""),
            pd.to_datetime(published.text if published is not None else None, utc=True, errors="coerce"),
        ))

    return records, []


def collect_html_source(source):
    records = []
    errors = []
    seen_urls = set()
    browser_headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/131.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
    }

    for landing_url in source["entrypoints"]:
        try:
            response = requests.get(landing_url, headers=browser_headers, timeout=REQUEST_TIMEOUT_SECONDS)
            response.raise_for_status()
        except Exception as exc:
            errors.append(f"{landing_url}: {type(exc).__name__}: {exc}")
            continue

        parser = LinkParser()
        parser.feed(response.text)
        landing_path = urlparse(landing_url).path.rstrip("/")
        for href, raw_title in parser.links:
            url = canonicalize_url(urljoin(landing_url, href))
            title = clean_html(raw_title)
            if (
                not source_url_is_allowed(source, url)
                or urlparse(url).path.rstrip("/") == landing_path
                or not title
                or len(title) < 12
                or url in seen_urls
            ):
                continue
            seen_urls.add(url)
            records.append(make_record(source, url, title))

    return records, errors


collection_errors = []
live_records = []
for source in SOURCE_REGISTRY:
    try:
        if source.get("collection_method") == "wordpress_api" and source.get("api_url"):
            live_records.extend(collect_wordpress_posts(source, collection_since))
        elif source.get("collection_method") == "rss_paginated" and source.get("feed_url"):
            source_records, source_errors = collect_rss_source(source)
            live_records.extend(source_records)
            collection_errors.extend(f"{source['publisher']}: {error}" for error in source_errors)
        else:
            source_records, source_errors = collect_html_source(source)
            live_records.extend(source_records)
            collection_errors.extend(f"{source['publisher']}: {error}" for error in source_errors)
    except Exception as exc:
        collection_errors.append(f"{source['publisher']}: {type(exc).__name__}: {exc}")

collection_error = " | ".join(collection_errors)
collection_status = "partial" if live_records and collection_errors else ("error" if collection_errors else "ok")
print(f"Collection status: {collection_status}")
print(f"Registered sources: {len(SOURCE_REGISTRY)}")
print(f"Fetched records: {len(live_records)}")
if collection_error:
    print(collection_error)


Collection status: partial
Registered sources: 11
Fetched records: 198
Financial Times: https://www.ft.com/business-education: HTTPError: 403 Client Error: Forbidden for url: https://www.ft.com/business-education | Bloomberg Businessweek: https://www.bloomberg.com/business-schools/: HTTPError: 403 Client Error: Forbidden for url: https://www.bloomberg.com/business-schools/


## 3. Apply transparent Important rules

These rules are intentionally simple and inspectable. They can be expanded after the online execution is proven reliable.


In [24]:
IMPORTANT_RULES = {
    "ranking_release": re.compile(r"\b(?:ranking|rankings|ranked|top\s+\d+)\b", re.I),
    "methodology_or_weights": re.compile(
        r"\b(?:ranking\s+methodology|methodology\s+and\s+weights|academic\s+reputation|employer\s+reputation|citations?\s+per\s+faculty|faculty[\s-]+student\s+ratio|international\s+faculty|international\s+students?|international\s+research\s+network|employment\s+outcomes?|sustainability\s+score|overall\s+score|indicator\s+weights?|metric\s+weights?|weighting\s+criteria|weighting\s+methodology|percentage\s+weights?|score\s+weights?|ranking\s+criteria)\b",
        re.I,
    ),
    "eligibility_or_deadline": re.compile(r"\b(?:eligibility|eligible|participation|submission|deadline)\b", re.I),
    "accreditation": re.compile(r"\b(?:accreditation|accredited|reaccredited|AACSB|AMBA|EQUIS|ZEvA)\b", re.I),
    "germany_or_DE": re.compile(r"\b(?:germany|de|german|deutschland)\b", re.I),
}
ESMT_PATTERN = re.compile(
    r"\b(?:ESMT(?:\s+Berlin)?|European\s+School\s+of\s+Management\s+(?:and|&)\s+Technology)\b",
    re.I,
)


def important_reasons(text):
    reasons = [name for name, pattern in IMPORTANT_RULES.items() if pattern.search(text)]
    if ESMT_PATTERN.search(text):
        reasons.insert(0, "esmt_mention")
    return reasons


new_frame = pd.DataFrame(live_records)
if not new_frame.empty:
    searchable = new_frame[["title", "excerpt"]].fillna("").agg(" ".join, axis=1)
    reason_lists = searchable.map(important_reasons)
    new_frame["important_reasons"] = reason_lists.map(lambda values: " | ".join(values))
    new_frame["is_important"] = reason_lists.map(bool)
else:
    new_frame = pd.DataFrame(columns=[
        "article_id", "source_id", "publisher", "source_group", "title", "excerpt",
        "url", "published_at", "collected_at", "important_reasons", "is_important",
    ])

display(new_frame[["published_at", "title", "is_important", "important_reasons"]].head(10))


,published_at,title,is_important,important_reasons
0,NaT,How universities are shaping ASEAN's tomorrow ...,False,
1,NaT,QS World Future Skills Index 2027 Mapping the ...,False,
2,NaT,Report 20 August 2026 Student mobility and mot...,False,
3,NaT,"Report 19 August 2026 Red, amber, green: What ...",False,
4,NaT,Article 18 August 2026 How important are gradu...,False,
5,NaT,Report 18 August 2026 学生流动与留学动机：中国 Student rec...,False,
6,NaT,Article 13 August 2026 International student r...,True,methodology_or_weights
7,NaT,Article 11 August 2026 When focusing on vocati...,False,
8,NaT,Report 10 August 2026 Shifts in Asian intrareg...,False,
9,NaT,Report 7 August 2026 The Emergence of the Augm...,False,


## 4. Merge, deduplicate and persist

The existing CSV remains the source of history. A daily run rechecks a small overlap, replaces duplicate URLs with the freshest copy, and then removes records older than 365 days.


In [22]:
EXPECTED_COLUMNS = [
    "article_id", "source_id", "publisher", "source_group", "title", "excerpt",
    "url", "published_at", "collected_at", "important_reasons", "is_important",
]

if NEWS_PATH.exists():
    existing_frame = pd.read_csv(NEWS_PATH)
    for date_column in ["published_at", "collected_at"]:
        existing_frame[date_column] = pd.to_datetime(existing_frame[date_column], utc=True, errors="coerce")
else:
    existing_frame = pd.DataFrame(columns=EXPECTED_COLUMNS)

existing_urls = set(existing_frame.get("url", pd.Series(dtype=str)).dropna())
new_item_count = int((~new_frame.get("url", pd.Series(dtype=str)).isin(existing_urls)).sum())

if existing_frame.empty:
    combined = new_frame.copy()
elif new_frame.empty:
    combined = existing_frame.copy()
else:
    combined = pd.concat([existing_frame, new_frame], ignore_index=True)
combined = combined.drop_duplicates("url", keep="last")
retention_cutoff = NOW_UTC - pd.Timedelta(days=RETENTION_DAYS)
is_important = combined["is_important"].astype(str).str.lower().eq("true")
combined = combined[
    is_important
    | combined["published_at"].isna()
    | (combined["published_at"] >= retention_cutoff)
].copy()
combined = combined.sort_values("published_at", ascending=False, na_position="last")
combined = combined[EXPECTED_COLUMNS]
combined.to_csv(NEWS_PATH, index=False)

status_frame = pd.DataFrame([{
    "run_at_utc": NOW_UTC.isoformat(),
    "source_id": SOURCE["source_id"],
    "publisher": SOURCE["publisher"],
    "status": collection_status,
    "collection_mode": collection_mode,
    "collection_since_utc": collection_since.isoformat(),
    "fetched_items": len(new_frame),
    "new_items": new_item_count,
    "stored_items": len(combined),
    "error": collection_error,
}])
status_frame.to_csv(STATUS_PATH, index=False)

if collection_status in {"ok", "partial"}:
    new_state = {
        "last_successful_run_utc": NOW_UTC.isoformat(),
        "last_collection_mode": collection_mode,
        "stored_items": len(combined),
    }
    STATE_PATH.write_text(json.dumps(new_state, indent=2), encoding="utf-8")

display(status_frame)
print(f"Stored articles: {len(combined)}")
print(f"New articles this run: {new_item_count}")
print(f"Important articles stored: {int(combined['is_important'].astype(str).str.lower().eq('true').sum())}")

if collection_status == "error":
    raise RuntimeError(f"Live collection failed: {collection_error}")


,run_at_utc,source_id,publisher,status,collection_mode,collection_since_utc,fetched_items,new_items,stored_items,error
0,2026-08-20T19:09:07.967243+00:00,corporate_knights,Corporate Knights,partial,incremental_with_48h_overlap,2026-08-18T19:07:09.656939+00:00,198,0,520,Financial Times: https://www.ft.com/business-e...


Stored articles: 520
New articles this run: 0
Important articles stored: 89


## What success looks like online

After the workflow runs successfully, GitHub should contain a new bot commit with:

- `data/news.csv`;
- `data/collector_status.csv`;
- `data/run_state.json`.

The first run records `initial_90_day_backfill`. The next run records `incremental_with_48h_overlap`. The Streamlit app reads these files directly; no API is required for this infrastructure test.
